# 03 - Final Model Training (Unified Harness)

Trains every model at its best hyperparameters (from Notebook 02) on the full
tuning pool, across 5 random seeds, using the SAME `fit_fn` interface for
every model -- ELinear and every NeuralForecast benchmark alike. This is a
direct continuation of Notebook 02's fairness commitment: not just the same
CV procedure, but the same final-training procedure and the same seed
protocol for every model in the comparison.

**Why 5 seeds:** deep models are sensitive to initialization; reporting
mean +/- std across seeds is standard practice and is required before any
Diebold-Mariano comparison (Notebook 04) is credible.

## Colab setup
Run this once per session. It mounts Google Drive (where the whole `potato_pipeline/` project folder should be uploaded), `cd`s into `notebooks/` so every relative path (`../src`, `../data`, `../results`) keeps working unchanged, and installs the extra packages this notebook needs (Colab's base image doesn't include them).

**GPU:** `Runtime -> Change runtime type -> T4 GPU` recommended for this notebook (deep learning training).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Adjust this if you uploaded the project folder somewhere else in Drive
PROJECT_ROOT = "/content/drive/MyDrive/potato_pipeline"
os.chdir(os.path.join(PROJECT_ROOT, "notebooks"))
print("Working directory:", os.getcwd())

In [ ]:
!pip install -q pmdarima neuralforecast

In [ ]:
import sys, os, json, shutil
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import torch
import warnings
warnings.filterwarnings("ignore")

import cv_utils
import elinear_model
import nf_wrappers

RESULTS_DIR = "../results"
PRED_DIR = "../results/predictions"
NF_CKPT_DIR = "../results/nf_checkpoints"
os.makedirs(PRED_DIR, exist_ok=True)
os.makedirs(NF_CKPT_DIR, exist_ok=True)

HORIZONS = [1, 7, 14, 30]
SEEDS = [0, 1, 2, 3, 4]
DL_MODEL_KEYS = ["ELinear", "LSTM", "TCN", "NBEATS", "DLinear", "NLinear", "PatchTST", "iTransformer"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

df = pd.read_csv("../data/agra_potato_clean.csv", index_col=0, parse_dates=True)
series = df["price"].values
dates = df.index
n = len(series)

pool_idx, test_idx = cv_utils.chronological_holdout(n, test_frac=0.15)
series_pool = series[pool_idx]
print(f"Tuning pool: {len(series_pool)} days | Test block: {len(test_idx)} days "
      f"({dates[test_idx[0]].date()} to {dates[test_idx[-1]].date()})")

## 1. Naive & Seasonal-Naive baselines (no training required)

- **Naive (persistence):** forecast = last observed value, repeated for all
  steps in the horizon.
- **Seasonal-Naive:** forecast = value observed exactly one seasonal period
  (365 days) prior. Standard, hard-to-beat baseline for seasonal commodity
  series (Hyndman & Athanasopoulos, *FPP3*, Ch. 5.2). Scored directly in
  Notebook 04 -- no training artifacts to save here.

## 2. Unified multi-seed training for every DL model

Every model -- ELinear and every NeuralForecast benchmark -- is trained
through the same `fit_fn(train_series, seq_len, pred_len, params, seed)`
signature established in Notebook 02. Checkpoints are saved per model per
horizon per seed:
- ELinear: PyTorch `state_dict` (`.pt`)
- NeuralForecast models: native `nf.save()` directories

In [ ]:
def get_fit_fn(model_key):
    if model_key == "ELinear":
        return elinear_model.elinear_fit_fn
    return nf_wrappers.make_nf_fit_fn(model_key)

# Normalize using stats from the FULL tuning pool (single final fit, not
# per-fold like Notebook 02's CV). Notebook 04 recomputes these same stats
# independently from series_pool -- deterministic given the same data, so
# no need to persist mean/std across notebooks.
pool_mean, pool_std = cv_utils.zscore_stats(series_pool)
series_pool_norm = cv_utils.zscore_transform(series_pool, pool_mean, pool_std)
print(f"Normalization stats (from tuning pool): mean={pool_mean:.2f}, std={pool_std:.2f}")

trained_meta = {}  # trained_meta[model_key][horizon] = {"seq_len": int, "seeds": [...]}

for model_key in DL_MODEL_KEYS:
    with open(f"{RESULTS_DIR}/best_hp_{model_key.lower()}.json") as f:
        hp = json.load(f)

    fit_fn = get_fit_fn(model_key)
    trained_meta[model_key] = {}

    for h_str, cfg in hp.items():
        h = int(h_str)
        params = cfg["params"]
        seq_len = params["seq_len"]

        for seed in SEEDS:
            model = fit_fn(series_pool_norm, seq_len, h, params, seed)

            if model_key == "ELinear":
                ckpt_path = f"{PRED_DIR}/elinear_h{h}_seed{seed}.pt"
                torch.save({"state_dict": model.state_dict(), "seq_len": seq_len,
                            "n_learners": params["n_learners"]}, ckpt_path)
            else:
                ckpt_path = f"{NF_CKPT_DIR}/{model_key}_h{h}_seed{seed}"
                model.save(ckpt_path, overwrite=True)

        trained_meta[model_key][h] = {"seq_len": seq_len, "seeds": SEEDS}
        print(f"[{model_key}] horizon={h}: trained {len(SEEDS)} seeds, seq_len={seq_len}")

with open(f"{RESULTS_DIR}/trained_meta.json", "w") as f:
    json.dump(trained_meta, f, indent=2)

## 3. ARIMA (deterministic -- single fit, no seed variance)

ARIMA has no stochastic initialization, so a single fit per horizon on the
full tuning pool is sufficient (unlike the DL models). Refit here (cheap)
rather than persisting a pickled model object, since `pmdarima` models are
lightweight to refit and pickling statsmodels objects across sessions can be
fragile.

In [ ]:
import pmdarima as pm
from pmdarima.preprocessing import FourierFeaturizer

with open(f"{RESULTS_DIR}/best_hp_arima.json") as f:
    arima_hp = json.load(f)

print("ARIMA hyperparameters were selected via auto_arima's internal AICc search "
      "(Notebook 02) -- refitting on the full tuning pool per horizon in Notebook 04 "
      "directly, since order selection is deterministic given the data.")
print(json.dumps(arima_hp, indent=2))

## 4. Save training artifacts summary

In [ ]:
print("Training complete. Artifacts:")
print(f"  ELinear checkpoints: {PRED_DIR}/elinear_h*_seed*.pt")
print(f"  NeuralForecast checkpoints: {NF_CKPT_DIR}/<model>_h*_seed*/")
print(f"  Metadata: {RESULTS_DIR}/trained_meta.json")
print("ARIMA and Naive/SeasonalNaive are refit/computed directly in Notebook 04 "
      "(cheap, deterministic -- no benefit to persisting checkpoints).")